# Part 1: RAG Pipeline

In [1]:
try:
    __import__('pysqlite3')
    import sys
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')
except ImportError:
    pass

import os
import json
import chromadb
import chromadb.utils.embedding_functions as embedding_functions
from dotenv import load_dotenv

load_dotenv()

True

## Initialize ChromaDB Client

In [2]:
chroma_client = chromadb.PersistentClient(path="chromadb")

## Create Collection with OpenAI Embedding

In [3]:
openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("OPENAI_API_KEY"),
    api_base=os.getenv("OPENAI_BASE_URL")
)

collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=openai_ef
)

## Load and Ingest Game Data

In [4]:
games_dir = "games"
for filename in os.listdir(games_dir):
    if filename.endswith(".json"):
        with open(os.path.join(games_dir, filename), "r") as f:
            game = json.load(f)
            
            # Format document string
            content = f"[{game.get('Platform', 'Unknown')}] {game.get('Name', 'Unknown')} ({game.get('YearOfRelease', 'Unknown')}) - {game.get('Description', '')}"
            doc_id = filename
            
            # Convert metadata values to strings
            meta = {k: str(v) for k, v in game.items() if v is not None}
            
            collection.add(
                ids=[doc_id],
                documents=[content],
                metadatas=[meta]
            )
print(f"Total documents in collection: {collection.count()}")

Total documents in collection: 15


## Verify: Semantic Search Demo

In [5]:
queries = [
    "racing game on PlayStation",
    "Nintendo fighting game",
    "open world shooter"
]

for query in queries:
    print(f"\n--- Query: {query} ---")
    results = collection.query(
        query_texts=[query],
        n_results=1
    )
    
    for doc, dist in zip(results["documents"][0], results["distances"][0]):
        print(f"Result: {doc}")
        print(f"Distance: {dist:.4f}")


--- Query: racing game on PlayStation ---


Result: [PlayStation 3] Gran Turismo 5 (2010) - A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.
Distance: 0.1166

--- Query: Nintendo fighting game ---


Result: [GameCube] Super Smash Bros. Melee (2001) - A crossover fighting game featuring characters from various Nintendo franchises battling it out in dynamic arenas.
Distance: 0.1627

--- Query: open world shooter ---


Result: [PlayStation 4] Marvel's Spider-Man (2018) - An open-world superhero game that lets players swing through New York City as Spider-Man, battling iconic villains.
Distance: 0.1795
